In [15]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))

In [ ]:
"""
Educational Goal:
- Why this module exists in an MLOps system: Centralizes feature engineering logic to prevent data leakage and ensure consistent preprocessing between training and inference.
- Responsibility (separation of concerns): Define feature transformation recipe only (no fitting, no data mutation).
- Pipeline contract (inputs and outputs): Configuration lists in → ColumnTransformer recipe out.
"""

from typing import Optional, List
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder


def get_feature_preprocessor(
    quantile_bin_cols: Optional[List[str]] = None,
    categorical_onehot_cols: Optional[List[str]] = None,
    numeric_passthrough_cols: Optional[List[str]] = None,
    n_bins: int = 3
):
    """
    Inputs:
    - quantile_bin_cols: numeric columns to apply quantile binning
    - categorical_onehot_cols: categorical columns for one-hot encoding
    - numeric_passthrough_cols: numeric columns to pass through unchanged
    - n_bins: number of bins for quantile discretization
    Outputs:
    - ColumnTransformer (unfitted)
    Why this contract matters for reliable ML delivery:
    - Guarantees consistent preprocessing during training and inference.
    - Prevents leakage by returning a blueprint without fitting.
    """

    print("Building feature preprocessing recipe...")  # TODO: replace with logging later

    quantile_bin_cols = quantile_bin_cols or []
    categorical_onehot_cols = categorical_onehot_cols or []
    numeric_passthrough_cols = numeric_passthrough_cols or []

    transformers = []

    # Quantile binning (numeric)
    if quantile_bin_cols:
        transformers.append(
            (
                "quantile_bin",
                KBinsDiscretizer(
                    n_bins=n_bins,
                    encode="ordinal",
                    strategy="quantile"
                ),
                quantile_bin_cols,
            )
        )

    # One-hot encoding (categorical)
    if categorical_onehot_cols:
        try:
            encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

        transformers.append(
            (
                "categorical_onehot",
                encoder,
                categorical_onehot_cols,
            )
        )

    # Numeric passthrough
    if numeric_passthrough_cols:
        transformers.append(
            (
                "numeric_passthrough",
                "passthrough",
                numeric_passthrough_cols,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    print("Warning: Custom feature logic not implemented yet")

    return preprocessor

In [9]:
import sklearn
print("sklearn works")

sklearn works


In [10]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [11]:
X_train[numeric_features].head()
X_train[categorical_features].head()

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
2532,Male,No,No,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic)
6654,Female,No,No,Yes,No,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check
4027,Male,Yes,Yes,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Bank transfer (automatic)
6795,Female,Yes,No,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check
808,Male,No,No,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Bank transfer (automatic)


In [16]:
from src.features import get_feature_preprocessor

preprocessor = get_feature_preprocessor(
    quantile_bin_cols=["num_feature"],
    categorical_onehot_cols=["cat_feature"],
    numeric_passthrough_cols=[]
)

print(preprocessor)

Building Telco feature preprocessing recipe...
ColumnTransformer(transformers=[('quantile_bin',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('kbins',
                                                  KBinsDiscretizer(encode='ordinal',
                                                                   n_bins=3))]),
                                 ['num_feature']),
                                ('categorical_onehot',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['